# NB-03: Feature Engineering — Trajectory, Verification & Behavioral Signals

**Goal:** Build the engineered feature set that will feed NB-05 (pseudo-label/tier assignment) and NB-06 (ranker training). This notebook does NOT assign tiers or scores — it only produces features, kept separate so tier-assignment logic (NB-05) can be iterated on independently without re-running feature engineering.

**Inputs:** `cleaned_candidates_v1.parquet` (NB-01), `jd_rule_flags.parquet` (NB-02)

**Output:** `features_structured.parquet` (candidate_id + all engineered features, joined with JD rule flags)

**Feature groups planned:**
1. Verified-vs-claimed skill gap (extends NB-01's `unverified_high_prof_count` into a cleaner composite)
2. Career trajectory (seniority slope, product-vs-services ratio, tenure stability)
3. Must-have production evidence detector (embeddings/retrieval/vector-DB work at a product company — the JD's actual stated bar, not just a skill tag)
4. Behavioral reliability composite (from the 23 signals)
5. Availability match (location, notice period, relocation vs Pune/Noida)
6. Education/institution tier (already built in NB-01, just carried through)

In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

NB01_OUT = Path("/kaggle/input/notebooks/kritikabenjwal/eda-and-schema-audit")   # adjust to your real Kaggle input path
NB02_OUT = Path("/kaggle/input/notebooks/kritikabenjwal/jd-rule-extraction")     # adjust to your real Kaggle input path

df_flat = pd.read_parquet(NB01_OUT / "cleaned_candidates_v1.parquet")
jd_rules = pd.read_parquet(NB02_OUT / "jd_rule_flags.parquet")

print("df_flat shape:", df_flat.shape)
print("jd_rules shape:", jd_rules.shape)

# Confirm 1:1 join on candidate_id before merging anything
print("\ncandidate_id match check:")
print("df_flat unique ids:", df_flat['candidate_id'].nunique())
print("jd_rules unique ids:", jd_rules['candidate_id'].nunique())
print("ids in both:", len(set(df_flat['candidate_id']) & set(jd_rules['candidate_id'])))

df = df_flat.merge(jd_rules, on='candidate_id', how='inner', validate='one_to_one')
print("\nMerged shape:", df.shape)
print("Columns:", list(df.columns))

df_flat shape: (100000, 51)
jd_rules shape: (100000, 10)

candidate_id match check:
df_flat unique ids: 100000
jd_rules unique ids: 100000
ids in both: 100000

Merged shape: (100000, 60)
Columns: ['candidate_id', 'profile_anonymized_name', 'profile_headline', 'profile_summary', 'profile_location', 'profile_country', 'profile_years_of_experience', 'profile_current_title', 'profile_current_company', 'profile_current_company_size', 'profile_current_industry', 'career_history', 'education', 'skills', 'certifications', 'languages', 'sig_profile_completeness_score', 'sig_signup_date', 'sig_last_active_date', 'sig_open_to_work_flag', 'sig_profile_views_received_30d', 'sig_applications_submitted_30d', 'sig_recruiter_response_rate', 'sig_avg_response_time_hours', 'sig_connection_count', 'sig_endorsements_received', 'sig_notice_period_days', 'sig_preferred_work_mode', 'sig_willing_to_relocate', 'sig_github_activity_score', 'sig_search_appearance_30d', 'sig_saved_by_recruiters_30d', 'sig_intervie

## Step 1: Rebuild the 44-template pool and tag templates for production retrieval/search evidence

NB-02 confirmed `career_history.description` is drawn from a fixed pool of 44 templates (not freely generated). Reuse that fact here: instead of keyword-scanning description text (which produced multiple false-positive funnels in NB-02), identify which of the 44 templates genuinely describe production embeddings/retrieval/search/recommendation/vector-DB work — this is the JD's single most important must-have signal, so it deserves the same care as the disqualifier rules, not a shortcut.

In [2]:
all_descriptions = []
for ch in df['career_history']:
    for job in ch:
        all_descriptions.append(job['description'])

unique_descriptions = sorted(set(all_descriptions))
print(f"Total unique templates: {len(unique_descriptions)}\n")

for i, desc in enumerate(unique_descriptions):
    print(f"--- TEMPLATE {i} ---")
    print(desc)
    print()

Total unique templates: 44

--- TEMPLATE 0 ---
Android mobile development using Java and (more recently) Kotlin at a consumer-app company. Built and maintained multiple production features including the main shopping flow, push notification system, and the offline-first sync layer. Comfortable with the Android framework, Jetpack components, and the typical patterns (MVVM, Hilt, Coroutines). My career has been entirely on mobile so far; interested in expanding into broader backend or platform engineering.

--- TEMPLATE 1 ---
Backend + data hybrid role at a growth-stage startup. Built the company's first proper data warehouse (migrating from a tangled set of Postgres replicas to a clean Snowflake setup with dbt), the orchestration layer (Airflow), and the BI integration (Looker). Shipped a couple of small predictive features but the bulk of the role was data infrastructure.

--- TEMPLATE 2 ---
Backend development with Python (FastAPI), PostgreSQL, and Redis at a B2B SaaS product. Owned t

## Step 2: Tag templates for production embeddings/retrieval/search/recommendation evidence

Manually classified all 44 templates against the JD's core must-have: "production experience with embeddings-based retrieval deployed to real users" and "production experience with vector DBs or hybrid search infra." 17 of 44 templates qualify — everything else (data infra without retrieval ownership, CV/NLP-classification-only, non-technical roles, and *integration-only* work like T2/T16 where someone else owns the model) does not meet this specific bar, even where it's otherwise solid engineering.

This becomes a template-ID lookup, not phrase-matching — avoids repeating the false-positive funnels from NB-02.

In [3]:
PRODUCTION_RETRIEVAL_TEMPLATE_IDS = {5, 6, 9, 11, 12, 19, 20, 22, 25, 28, 29, 34, 35, 36, 37, 39, 41}

# Stronger sub-tier: templates with explicit measured eval (NDCG/A-B/recall/latency numbers), the JD's exact "ranking evaluation experience" ask
STRONG_RETRIEVAL_TEMPLATE_IDS = {5, 20, 29, 34, 35}

template_to_id = {desc: i for i, desc in enumerate(unique_descriptions)}

def get_retrieval_evidence(career_history):
    """Returns (has_any_retrieval_evidence, has_strong_retrieval_evidence, matching_job_titles)"""
    has_any = False
    has_strong = False
    matches = []
    for job in career_history:
        tid = template_to_id.get(job['description'])
        if tid in PRODUCTION_RETRIEVAL_TEMPLATE_IDS:
            has_any = True
            matches.append(job['title'])
        if tid in STRONG_RETRIEVAL_TEMPLATE_IDS:
            has_strong = True
    return pd.Series({
        'must_have_retrieval_evidence': has_any,
        'must_have_retrieval_evidence_strong': has_strong,
        'retrieval_evidence_job_titles': matches
    })

retrieval_features = df['career_history'].apply(get_retrieval_evidence)
df = pd.concat([df, retrieval_features], axis=1)

print("Candidates with ANY production retrieval/search/recommendation evidence:", df['must_have_retrieval_evidence'].sum())
print("Candidates with STRONG (measured-eval) retrieval evidence:", df['must_have_retrieval_evidence_strong'].sum())

print("\n--- Sample: strong evidence candidates ---")
sample = df[df['must_have_retrieval_evidence_strong']].head(8)
for idx, row in sample.iterrows():
    print(f"{row['candidate_id']} | title: {row['profile_current_title']} | yoe: {row['profile_years_of_experience']} | evidence roles: {row['retrieval_evidence_job_titles']}")

print("\n--- Cross-check: overlap with genuine ML/AI-titled population (is_ml_ai_practitioner_by_title from NB-02 wasn't carried over — rebuild quickly here) ---")
print("Note: this feature is title-independent by design — retrieval evidence should be checkable regardless of exact job title, since the JD explicitly says the specific tool/title doesn't matter, the operational experience does.")

Candidates with ANY production retrieval/search/recommendation evidence: 504
Candidates with STRONG (measured-eval) retrieval evidence: 75

--- Sample: strong evidence candidates ---
CAND_0001610 | title: Machine Learning Engineer | yoe: 3.0 | evidence roles: ['Machine Learning Engineer', 'Senior Data Scientist', 'Machine Learning Engineer']
CAND_0003977 | title: Recommendation Systems Engineer | yoe: 4.6 | evidence roles: ['Recommendation Systems Engineer', 'NLP Engineer']
CAND_0005260 | title: Senior NLP Engineer | yoe: 5.2 | evidence roles: ['Senior NLP Engineer', 'Senior NLP Engineer']
CAND_0005649 | title: Senior Data Scientist | yoe: 7.4 | evidence roles: ['Senior Data Scientist', 'Recommendation Systems Engineer', 'Search Engineer']
CAND_0007009 | title: Recommendation Systems Engineer | yoe: 7.9 | evidence roles: ['Recommendation Systems Engineer', 'Recommendation Systems Engineer', 'Recommendation Systems Engineer']
CAND_0007411 | title: Senior Machine Learning Engineer | yoe:

## Step 3: Career trajectory features — seniority slope, product-vs-services ratio, tenure stability

Three trajectory features per the doc's section 5 plan:
1. **Seniority slope** — is the candidate moving up over time (junior→senior) or flat/declining, using the `seniority_rank` ordinal already built in NB-02
2. **Product-vs-services ratio** — fraction of career spent at product companies vs IT-services/consulting firms (reuses `CONSULTING_FIRMS` from NB-02, but as a continuous ratio, not just the binary consulting-only flag)
3. **Tenure stability** — average months per role, weighted toward recent roles, as a job-hopping signal (distinct from the title-chasing soft-negative, which specifically requires seniority escalation — this is a general stability measure)

In [4]:
SENIORITY_RANK = {
    'principal': 5, 'director': 5, 'head of': 5,
    'staff': 4, 'architect': 4,
    'lead': 3, 'senior': 3,
}
CONSULTING_FIRMS = {'TCS', 'Infosys', 'Wipro', 'Accenture', 'Cognizant', 'Capgemini'}

def seniority_rank(title):
    title_lower = title.lower()
    for marker, rank in sorted(SENIORITY_RANK.items(), key=lambda x: -x[1]):
        if marker in title_lower:
            return rank
    return 1

def seniority_slope(career_history):
    """Linear trend of seniority rank over chronologically sorted roles. Positive = climbing."""
    if len(career_history) < 2:
        return 0.0
    sorted_jobs = sorted(career_history, key=lambda j: j['start_date'])
    ranks = [seniority_rank(j['title']) for j in sorted_jobs]
    x = np.arange(len(ranks))
    slope = np.polyfit(x, ranks, 1)[0]
    return float(slope)

def product_company_ratio(career_history):
    """Fraction of total career months spent at non-consulting (product) companies."""
    total_months = sum(j.get('duration_months', 0) for j in career_history)
    if total_months == 0:
        return 0.0
    product_months = sum(j.get('duration_months', 0) for j in career_history if j['company'] not in CONSULTING_FIRMS)
    return product_months / total_months

def avg_tenure_months(career_history):
    durations = [j.get('duration_months', 0) for j in career_history]
    return float(np.mean(durations)) if durations else 0.0

def recent_tenure_months(career_history):
    """Duration of the current/most recent role specifically -- short recent tenure is a stronger stability signal than average."""
    current = [j for j in career_history if j.get('is_current')]
    if current:
        return current[0].get('duration_months', 0)
    sorted_jobs = sorted(career_history, key=lambda j: j['start_date'])
    return sorted_jobs[-1].get('duration_months', 0)

df['feat_seniority_slope'] = df['career_history'].apply(seniority_slope)
df['feat_product_company_ratio'] = df['career_history'].apply(product_company_ratio)
df['feat_avg_tenure_months'] = df['career_history'].apply(avg_tenure_months)
df['feat_recent_tenure_months'] = df['career_history'].apply(recent_tenure_months)

for col in ['feat_seniority_slope', 'feat_product_company_ratio', 'feat_avg_tenure_months', 'feat_recent_tenure_months']:
    print(f"--- {col} ---")
    print(df[col].describe())
    print()

print("Correlation of trajectory features with years_of_experience (sanity check, expect weak-moderate, not near-1.0):")
print(df[['feat_seniority_slope','feat_product_company_ratio','feat_avg_tenure_months','feat_recent_tenure_months','profile_years_of_experience']].corr()['profile_years_of_experience'])

--- feat_seniority_slope ---
count    1.000000e+05
mean    -1.575000e-03
std      2.185944e-01
min     -2.000000e+00
25%     -8.214890e-17
50%     -7.850462e-17
75%      7.951207e-18
max      2.000000e+00
Name: feat_seniority_slope, dtype: float64

--- feat_product_company_ratio ---
count    100000.000000
mean          0.736693
std           0.310402
min           0.000000
25%           0.550725
50%           0.836112
75%           1.000000
max           1.000000
Name: feat_product_company_ratio, dtype: float64

--- feat_avg_tenure_months ---
count    100000.000000
mean         28.361075
std           7.819599
min           8.000000
25%          23.250000
50%          28.000000
75%          33.000000
max          98.000000
Name: feat_avg_tenure_months, dtype: float64

--- feat_recent_tenure_months ---
count    100000.000000
mean         30.710620
std          12.022822
min           8.000000
25%          20.000000
50%          30.000000
75%          40.000000
max         228.000000
Nam

## Step 4: Verified-vs-claimed skill gap (normalized) + behavioral reliability composite

**Skill gap** — NB-01 built the raw counts (`claimed_high_prof_skill_count`, `verified_skill_count`, `unverified_high_prof_count`). Turn these into one normalized, model-ready feature: what fraction of a candidate's *claimed* high-proficiency skills are backed by zero verification. This is the JD's single strongest anti-keyword-stuffing signal, so it deserves its own clean composite rather than three raw counts left for the model to figure out.

**Behavioral reliability composite** — weighted blend of the trust/reliability-flavored signals from the 23 behavioral fields: `recruiter_response_rate`, `interview_completion_rate`, `offer_acceptance_rate` (sentinel-aware), recency of `last_active_date`, plus the verification booleans (`verified_email`, `verified_phone`, `linkedin_connected`). This is NOT the availability-match feature (notice period, relocation) — that's a separate step, since availability and reliability answer different questions per the JD's own framing ("down-weight, don't disqualify").

In [5]:
# --- Skill gap ratio ---
def unverified_claim_ratio(row):
    if row['claimed_high_prof_skill_count'] == 0:
        return 0.0  # no high-prof claims at all -> nothing to be unverified
    return row['unverified_high_prof_count'] / row['claimed_high_prof_skill_count']

df['feat_unverified_claim_ratio'] = df.apply(unverified_claim_ratio, axis=1)
print("--- feat_unverified_claim_ratio ---")
print(df['feat_unverified_claim_ratio'].describe())

# --- Behavioral reliability composite ---
import datetime

# last_active_date recency (days since, smaller = more recent = more reliable)
df['sig_last_active_date'] = pd.to_datetime(df['sig_last_active_date'])
REFERENCE_DATE = df['sig_last_active_date'].max()  # use dataset's own max date as "today" proxy, avoids real-world clock drift
df['_days_since_active'] = (REFERENCE_DATE - df['sig_last_active_date']).dt.days
print("\n--- days_since_active (reference = dataset max date) ---")
print(df['_days_since_active'].describe())

# Normalize recency into 0-1 (1 = most recent, decays over ~180 days)
df['_recency_score'] = np.clip(1 - (df['_days_since_active'] / 180), 0, 1)

# offer_acceptance_rate is sentinel-aware (-1 = no history) -- treat "no history" as neutral (0.5), not penalized
df['_offer_accept_clean'] = df['sig_offer_acceptance_rate'].where(df['sig_offer_acceptance_rate'] != -1, 0.5)

verification_bonus = (
    df['sig_verified_email'].astype(int) +
    df['sig_verified_phone'].astype(int) +
    df['sig_linkedin_connected'].astype(int)
) / 3.0

df['feat_behavioral_reliability'] = (
    0.25 * df['sig_recruiter_response_rate'] +
    0.25 * df['sig_interview_completion_rate'] +
    0.20 * df['_offer_accept_clean'] +
    0.20 * df['_recency_score'] +
    0.10 * verification_bonus
)

print("\n--- feat_behavioral_reliability ---")
print(df['feat_behavioral_reliability'].describe())

print("\nCorrelation with raw inputs (sanity check -- should correlate moderately with each, not dominated by one):")
print(df[['feat_behavioral_reliability','sig_recruiter_response_rate','sig_interview_completion_rate','_offer_accept_clean','_recency_score']].corr()['feat_behavioral_reliability'])

df = df.drop(columns=['_days_since_active', '_recency_score', '_offer_accept_clean'])

--- feat_unverified_claim_ratio ---
count    100000.000000
mean          0.257577
std           0.417254
min           0.000000
25%           0.000000
50%           0.000000
75%           0.500000
max           1.000000
Name: feat_unverified_claim_ratio, dtype: float64

--- days_since_active (reference = dataset max date) ---
count    100000.000000
mean        108.723890
std          66.588459
min           0.000000
25%          52.000000
50%         105.000000
75%         162.000000
max         240.000000
Name: _days_since_active, dtype: float64

--- feat_behavioral_reliability ---
count    100000.000000
mean          0.503593
std           0.107318
min           0.130000
25%           0.427222
50%           0.502722
75%           0.578611
max           0.901722
Name: feat_behavioral_reliability, dtype: float64

Correlation with raw inputs (sanity check -- should correlate moderately with each, not dominated by one):
feat_behavioral_reliability      1.000000
sig_recruiter_response_rat

## Step 5: Availability match features — location, relocation, notice period vs JD requirements

JD requirement: Pune/Noida based, or willing to relocate. Notice period preference: shorter is better, "still in scope but bar gets higher" beyond 30 days (a soft down-weight, not a disqualifier, per JD section 2).

Three features:
1. `feat_location_match` — categorical→ordinal: already in Pune/Noida (best), elsewhere in India + willing to relocate (good), elsewhere in India + not willing (weak), outside India entirely (weakest, regardless of relocation flag — cross-border relocation for this kind of role is a materially different ask)
2. `feat_notice_period_score` — continuous, inverted and normalized (shorter notice = higher score), matching the JD's own stated preference curve
3. `feat_work_mode_match` — placeholder check against JD's stated mode preference once we confirm what `sig_preferred_work_mode` values exist and what the JD actually asks for (need to check this against the JD text before hardcoding a match rule)

In [6]:
TARGET_LOCATIONS = {'Pune', 'Noida'}  # per JD

def location_match_score(row):
    location = row['profile_location']
    country = row['profile_country']
    willing_to_relocate = row['sig_willing_to_relocate']

    if country != 'India':
        return 0.25 if willing_to_relocate else 0.0  # cross-border relocation, low regardless

    # India-based: check if location string contains Pune or Noida
    in_target_city = any(city in location for city in TARGET_LOCATIONS)
    if in_target_city:
        return 1.0
    elif willing_to_relocate:
        return 0.7
    else:
        return 0.4  # India-based, not in target city, not willing -- still domestic, softest penalty

df['feat_location_match'] = df.apply(location_match_score, axis=1)
print("--- feat_location_match distribution ---")
print(df['feat_location_match'].value_counts().sort_index())

# Notice period: JD prefers shorter, sub-30-days ideal, gets "harder" beyond 30, quantized at 30/60/90/120/150
def notice_period_score(days):
    if days <= 30:
        return 1.0
    elif days <= 60:
        return 0.7
    elif days <= 90:
        return 0.45
    elif days <= 120:
        return 0.25
    else:
        return 0.1

df['feat_notice_period_score'] = df['sig_notice_period_days'].apply(notice_period_score)
print("\n--- feat_notice_period_score distribution ---")
print(df['feat_notice_period_score'].value_counts().sort_index())

# Check what preferred_work_mode values actually exist before building a match rule
print("\n--- sig_preferred_work_mode unique values ---")
print(df['sig_preferred_work_mode'].value_counts())

--- feat_location_match distribution ---
feat_location_match
0.00    17756
0.25     7131
0.40    47464
0.70    19180
1.00     8469
Name: count, dtype: int64

--- feat_notice_period_score distribution ---
feat_notice_period_score
0.10    12981
0.25    17570
0.45    31097
0.70    24543
1.00    13809
Name: count, dtype: int64

--- sig_preferred_work_mode unique values ---
sig_preferred_work_mode
hybrid      25076
onsite      25000
flexible    25000
remote      24924
Name: count, dtype: int64


## Step 6: Correcting feat_location_match against the real JD text — and dropping feat_work_mode_match

Two corrections from reading the literal JD:

1. **No work-mode preference exists in the JD** — it explicitly says "we don't require any specific number of in-office days." Building `feat_work_mode_match` would be inventing a signal that isn't there. **Dropped.**
2. **The accepted-location list is wider than Pune/Noida** — JD explicitly welcomes Hyderabad, Mumbai, Delhi NCR in addition to Pune/Noida, with "hybrid — flexible cadence" as the actual work arrangement. `feat_location_match` needs an extra tier for these welcomed-but-not-preferred cities, distinct from "elsewhere in India" or "outside India."

Corrected tier structure:
- Pune/Noida (explicitly preferred) → 1.0
- Hyderabad/Mumbai/Delhi NCR (explicitly welcomed) → 0.85
- Elsewhere in India + willing to relocate → 0.6
- Elsewhere in India + not willing → 0.35
- Outside India + willing to relocate → 0.2 (JD: "case-by-case," "don't sponsor work visas" — real friction, not equivalent to domestic relocation)
- Outside India + not willing → 0.0

In [7]:
PREFERRED_LOCATIONS = {'Pune', 'Noida'}
WELCOMED_LOCATIONS = {'Hyderabad', 'Mumbai', 'Delhi', 'NCR', 'Gurgaon', 'Gurugram'}  # Delhi NCR commonly appears as Delhi/Gurgaon/Gurugram in location strings

def location_match_score(row):
    location = row['profile_location']
    country = row['profile_country']
    willing_to_relocate = row['sig_willing_to_relocate']

    if country != 'India':
        return 0.2 if willing_to_relocate else 0.0

    if any(city in location for city in PREFERRED_LOCATIONS):
        return 1.0
    elif any(city in location for city in WELCOMED_LOCATIONS):
        return 0.85
    elif willing_to_relocate:
        return 0.6
    else:
        return 0.35

df['feat_location_match'] = df.apply(location_match_score, axis=1)
print("--- feat_location_match distribution (corrected) ---")
print(df['feat_location_match'].value_counts().sort_index())

# Confirm which raw location strings are landing in each tier, to catch string-format mismatches early
print("\n--- Sample profile_location values by tier ---")
for tier_val in sorted(df['feat_location_match'].unique()):
    sample_locs = df[df['feat_location_match'] == tier_val]['profile_location'].unique()[:5]
    print(f"tier {tier_val}: {list(sample_locs)}")

# Full location value counts, so we can visually confirm no welcomed city is being missed due to string formatting
print("\n--- All unique profile_location values ---")
print(df['profile_location'].value_counts())

--- feat_location_match distribution (corrected) ---
feat_location_match
0.00    17756
0.20     7131
0.35    35681
0.60    14439
0.85    16524
1.00     8469
Name: count, dtype: int64

--- Sample profile_location values by tier ---
tier 0.0: ['Toronto', 'Austin', 'New York', 'London', 'Dubai']
tier 0.2: ['Sydney', 'Austin', 'Dubai', 'Berlin', 'London']
tier 0.35: ['Chennai, Tamil Nadu', 'Chandigarh, Chandigarh', 'Trivandrum, Kerala', 'Bangalore, Karnataka', 'Bhubaneswar, Odisha']
tier 0.6: ['Bhubaneswar, Odisha', 'Trivandrum, Kerala', 'Coimbatore, Tamil Nadu', 'Chandigarh, Chandigarh', 'Indore, Madhya Pradesh']
tier 0.85: ['Gurgaon, Haryana', 'Hyderabad, Telangana', 'Delhi, Delhi', 'Mumbai, Maharashtra']
tier 1.0: ['Noida, Uttar Pradesh', 'Pune, Maharashtra']

--- All unique profile_location values ---
profile_location
Bhubaneswar, Odisha       4321
Hyderabad, Telangana      4283
Noida, Uttar Pradesh      4283
Jaipur, Rajasthan         4268
Bangalore, Karnataka      4238
Kolkata, West B

## Step 7: Finalize NB-03 — assemble features_structured.parquet

Consolidate every engineered feature built in this notebook, plus the carried-through NB-01/NB-02 columns needed downstream (institution tier, disqualifier flags, soft negatives), into one output. Nested/raw columns (career_history, skills, etc.) are dropped here — NB-05 and NB-06 should work off engineered features only, not re-parse raw JSON.

In [8]:
FEATURE_COLUMNS = [
    'candidate_id',
    # Profile passthrough (needed for reasoning generation later, NB-07)
    'profile_years_of_experience', 'profile_current_title', 'profile_current_company',
    'profile_current_industry', 'profile_location', 'profile_country',
    # Trajectory features (Step 3)
    'feat_seniority_slope', 'feat_product_company_ratio',
    'feat_avg_tenure_months', 'feat_recent_tenure_months',
    # Must-have production retrieval evidence (Step 2) -- the JD's core bar
    'must_have_retrieval_evidence', 'must_have_retrieval_evidence_strong', 'retrieval_evidence_job_titles',
    # Verified-vs-claimed skill gap (Step 4)
    'feat_unverified_claim_ratio', 'claimed_high_prof_skill_count', 'verified_skill_count',
    # Behavioral reliability (Step 4)
    'feat_behavioral_reliability',
    # Availability match (Step 5-6)
    'feat_location_match', 'feat_notice_period_score',
    # Carried through from NB-01
    'best_institution_tier_score', 'is_honeypot_candidate',
    'sig_has_github', 'sig_github_activity_score',
    # Carried through from NB-02 (JD rules)
    'is_hard_disqualified', 'hard_disq_langchain_wrapper_only',
    'hard_disq_consulting_only', 'hard_disq_closed_source_only',
    'soft_neg_title_chasing_score', 'soft_neg_framework_tutorial_ratio',
]

features_structured = df[FEATURE_COLUMNS].copy()

print("features_structured shape:", features_structured.shape)
print("\nNull check:")
print(features_structured.isnull().sum().sum(), "total nulls across all columns")
print("\nDtypes:")
print(features_structured.dtypes)

OUT_DIR = Path("/kaggle/working")
OUT_DIR.mkdir(exist_ok=True)
features_structured.to_parquet(OUT_DIR / "features_structured.parquet", index=False)
print("\nSaved features_structured.parquet")
print("Size on disk:", (OUT_DIR / "features_structured.parquet").stat().st_size / 1024, "KB")

print("\n--- Final feature summary stats ---")
print(features_structured.describe(include='all').T)

features_structured shape: (100000, 30)

Null check:
0 total nulls across all columns

Dtypes:
candidate_id                            object
profile_years_of_experience            float64
profile_current_title                   object
profile_current_company                 object
profile_current_industry                object
profile_location                        object
profile_country                         object
feat_seniority_slope                   float64
feat_product_company_ratio             float64
feat_avg_tenure_months                 float64
feat_recent_tenure_months                int64
must_have_retrieval_evidence              bool
must_have_retrieval_evidence_strong       bool
retrieval_evidence_job_titles           object
feat_unverified_claim_ratio            float64
claimed_high_prof_skill_count            int64
verified_skill_count                     int64
feat_behavioral_reliability            float64
feat_location_match                    float64
feat_notice_

## NB-03 Complete — Summary

**Output:** `features_structured.parquet` (100,000 rows × 30 columns, 0 nulls, 2.1MB)

**Feature groups built:**
| Group | Columns | Key finding |
|---|---|---|
| Trajectory | `feat_seniority_slope`, `feat_product_company_ratio`, `feat_avg_tenure_months`, `feat_recent_tenure_months` | Weak correlation with YoE (max 0.34) — genuinely new signal, not a YoE proxy |
| Must-have retrieval evidence | `must_have_retrieval_evidence(_strong)`, `retrieval_evidence_job_titles` | Template-ID lookup (not keyword matching) — 504 any / 75 strong, clean sample, no false-positive funnel needed |
| Skill verification gap | `feat_unverified_claim_ratio` | Mean 0.26, expected floor-at-zero shape |
| Behavioral reliability | `feat_behavioral_reliability` | Balanced composite, no single input dominates |
| Availability | `feat_location_match`, `feat_notice_period_score` | Corrected against literal JD text — 6-tier location score including Hyderabad/Mumbai/Delhi NCR as "welcomed," work-mode feature dropped entirely (JD states no preference) |
| Carried through | institution tier, honeypot flag, GitHub signals, all NB-02 disqualifier/soft-negative columns | — |

**Process note worth keeping for the Stage 5 defense:** this notebook corrected two assumptions against the literal JD text rather than guessing (location tiering, work-mode feature) — consistent with the NB-02 pattern of verifying against source text before trusting a heuristic.

**Next:** NB-04 — Semantic Embeddings (SBERT encode candidate text + the single fixed JD text, cosine similarity as one more feature — not the primary signal, per section 3 of the doc).